# Silver 02 — Enrich silver_replay_header with Player Nicknames

Reads rows from `silver_replay_header` where `account_id` is set but `player_nickname`
is empty, looks up the display names via the Trackmania OAuth API in batches,
and updates the table using a MERGE statement.

**Prerequisites:**
- A registered OAuth app at https://api.trackmania.com (gives you `client_id` + `client_secret`)
- `silver_replay_header` Delta table must already exist (run Silver_01_ghost_ingest first)

## Parameters

- `TM_CLIENT_ID` — Your Trackmania OAuth app Identifier
- `TM_CLIENT_SECRET` — Your Trackmania OAuth app Secret
- `BATCH_SIZE` — How many account IDs to look up per API call (max 50 per API docs)

In [ ]:
TM_CLIENT_ID     = ""   # Your OAuth app Identifier
TM_CLIENT_SECRET = ""   # Your OAuth app Secret

BATCH_SIZE = 50

## Authenticate with Trackmania OAuth API

Uses the `client_credentials` grant to obtain a bearer token from the Trackmania OAuth API.

In [ ]:
import requests

USER_AGENT = "tm2020-gbx-parser / fabric-enrich-nicknames / contact via GitHub"

try:
    token_resp = requests.post(
        "https://api.trackmania.com/api/access_token",
        headers={"User-Agent": USER_AGENT},
        data={
            "grant_type": "client_credentials",
            "client_id": TM_CLIENT_ID,
            "client_secret": TM_CLIENT_SECRET,
        },
    )
    token_resp.raise_for_status()
except Exception as e:
    raise RuntimeError(f"OAuth token request failed — check TM_CLIENT_ID / TM_CLIENT_SECRET. Details: {e}")

access_token = token_resp.json()["access_token"]

API_HEADERS = {
    "Authorization": f"Bearer {access_token}",
    "User-Agent": USER_AGENT,
}
print("✓ OAuth token obtained")

## Find Rows That Need a Nickname

Queries `silver_replay_header` for distinct `account_id` values where `player_nickname` is empty.

In [ ]:
df = spark.table("silver_replay_header")

missing = (
    df.filter((df.account_id != "") & (df.player_nickname == ""))
      .select("account_id")
      .distinct()
      .rdd.flatMap(lambda r: [r.account_id])
      .collect()
)

print(f"Found {len(missing)} account(s) with no nickname")

## Look Up Display Names in Batches

Calls the Trackmania display-names API in batches of up to `BATCH_SIZE` account IDs.

In [ ]:
DISPLAY_NAMES_URL = "https://api.trackmania.com/api/display-names"

nicknames = {}

for i in range(0, len(missing), BATCH_SIZE):
    batch = missing[i:i + BATCH_SIZE]
    try:
        response = requests.get(
            DISPLAY_NAMES_URL,
            headers=API_HEADERS,
            params=[("accountId[]", aid) for aid in batch],
        )
        response.raise_for_status()
        for account_id, display_name in response.json().items():
            nicknames[account_id] = display_name
        print(f"  Batch {i // BATCH_SIZE + 1}: looked up {len(batch)} account(s)")
    except Exception as e:
        print(f"  ✗ Batch {i // BATCH_SIZE + 1} failed: {e} — skipping")

print(f"\n✓ Resolved {len(nicknames)} nickname(s)")

## Update silver_replay_header

Merges resolved nicknames back into `silver_replay_header` using a Spark SQL MERGE statement.
Only rows with an empty `player_nickname` are updated.

In [ ]:
if nicknames:
    updates = spark.createDataFrame(
        [{"account_id": k, "player_nickname": v} for k, v in nicknames.items()]
    )
    updates.createOrReplaceTempView("nickname_updates")

    spark.sql("""
        MERGE INTO silver_replay_header AS target
        USING nickname_updates AS src
        ON target.account_id = src.account_id
        WHEN MATCHED AND target.player_nickname = ''
            THEN UPDATE SET target.player_nickname = src.player_nickname
    """)

    print(f"✓ Updated {len(nicknames)} row(s) in silver_replay_header")
else:
    print("Nothing to update")

## Summary

In [ ]:
print("\nUpdated rows:")
spark.table("silver_replay_header") \
    .filter("account_id != '' AND player_nickname != ''") \
    .select("replay_id", "account_id", "player_nickname", "map_uid") \
    .show(20, truncate=False)